In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Tạo thư mục ở project root thay vì trong notebooks/
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [6]:
import sys
import importlib

# Lùi lại 1 thư mục để truy cập vào src/models/
sys.path.append('../src/models')

# 1. ÉP PYTHON NẠP LẠI (RELOAD) FILE CONFIG VÀ ARCHITECTURE
import model_config_hands_pose
import architectures_hands_pose

importlib.reload(model_config_hands_pose)
importlib.reload(architectures_hands_pose)

# 2. Import trực tiếp các biến và hàm sau khi đã reload
from model_config_hands_pose import INPUT_SHAPE, NUM_CLASSES
from architectures_hands_pose import build_lstm_model, build_bilstm_model, build_cnn1d_model

print(f"Cấu hình TỪ POSE+HANDS CONFIG: INPUT_SHAPE={INPUT_SHAPE}, NUM_CLASSES={NUM_CLASSES}")

Cấu hình TỪ POSE+HANDS CONFIG: INPUT_SHAPE=(30, 258), NUM_CLASSES=30


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_and_print(model, model_name, X_test, y_test):
    # Lấy dự đoán từ mô hình
    y_pred_probs = model.predict(X_test, verbose=0)
    # Chuyển đổi xác suất thành nhãn (Label Encoding)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Tính toán các chỉ số đánh giá
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # In kết quả theo format định dạng sẵn
    print("=================================================================")
    print(f"           {model_name.upper()} EVALUATION RESULTS (TEST SET)")
    print("=================================================================")
    print(f"Accuracy:          {acc * 100:.2f}%")
    print(f"Macro Precision:   {precision:.4f}")
    print(f"Macro Recall:      {recall:.4f}")
    print(f"Macro F1-score:    {f1:.4f}")
    print("=================================================================")

In [8]:
# 3. Load Data từ thư mục pose_hands
data_dir = "../data/pose_hands/" 
X_train = np.load(f"{data_dir}X_train_aug.npy")
y_train = np.load(f"{data_dir}y_train_aug.npy")
X_val = np.load(f"{data_dir}X_val.npy")
y_val = np.load(f"{data_dir}y_val.npy")
X_test = np.load(f"{data_dir}X_test.npy")
y_test = np.load(f"{data_dir}y_test.npy")

print("\n--- KIỂM TRA SHAPE DỮ LIỆU ---")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# Đảm bảo nhãn đang ở dạng mảng 1 chiều (Label Encoding)
assert len(y_train.shape) == 1, "LỖI: Nhãn không phải Label Encoding 1 chiều!"
print("\n-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!")


--- KIỂM TRA SHAPE DỮ LIỆU ---
X_train shape: (3000, 30, 258)
y_train shape: (3000,)

-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!


In [9]:
# CẤU HÌNH HUẤN LUYỆN CHUNG
EPOCHS = 50
BATCH_SIZE = 32

In [10]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN LSTM (HANDS_POSE) ---")
lstm_model = build_lstm_model()

lstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Cập nhật tên file thành lstm_model_hands_pose.h5
    ModelCheckpoint('../models/lstm_model_hands_pose.h5', monitor='val_loss', save_best_only=True)
]

lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=lstm_callbacks,
    verbose=1
)

# ĐÃ SỬA: Cập nhật tên file thành lstm_history_hands_pose.json
with open('../results/lstm_history_hands_pose.json', 'w') as f:
    json.dump(lstm_history.history, f)

print(f"\n[*] LSTM training completed.")
print(f"[*] Final validation accuracy: {lstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/lstm_model_hands_pose.h5")


--- XÂY DỰNG & HUẤN LUYỆN LSTM (HANDS_POSE) ---


Epoch 1/50


94/94 [==============================] - 15s 83ms/step - loss: 2.9402 - accuracy: 0.1520 - val_loss: 2.1745 - val_accuracy: 0.3571
Epoch 2/50
 1/94 [..............................] - ETA: 5s - loss: 2.4494 - accuracy: 0.1562

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 6s 59ms/step - loss: 2.0196 - accuracy: 0.3743 - val_loss: 1.4834 - val_accuracy: 0.5190
Epoch 3/50
94/94 [==============================] - 5s 58ms/step - loss: 1.5797 - accuracy: 0.5020 - val_loss: 1.2110 - val_accuracy: 0.6571
Epoch 4/50
94/94 [==============================] - 5s 50ms/step - loss: 1.1545 - accuracy: 0.6650 - val_loss: 1.0164 - val_accuracy: 0.6762
Epoch 5/50
94/94 [==============================] - 5s 50ms/step - loss: 0.9582 - accuracy: 0.7057 - val_loss: 0.9044 - val_accuracy: 0.7333
Epoch 6/50
94/94 [==============================] - 5s 57ms/step - loss: 0.8615 - accuracy: 0.7460 - val_loss: 0.8013 - val_accuracy: 0.7333
Epoch 7/50
94/94 [==============================] - 5s 54ms/step - loss: 0.8090 - accuracy: 0.7660 - val_loss: 0.8869 - val_accuracy: 0.6857
Epoch 8/50
94/94 [==============================] - 5s 56ms/step - loss: 0.6575 - accuracy: 0.8003 - val_loss: 0.7836 - val_accuracy: 0.7714
Epoch 9/50
94/94 [======

In [11]:
# GỌI HÀM ĐÁNH GIÁ (Xuất bảng kết quả Test Set)
evaluate_and_print(lstm_model, "LSTM (HANDS_POSE)", X_test, y_test)

           LSTM (HANDS_POSE) EVALUATION RESULTS (TEST SET)
Accuracy:          84.36%
Macro Precision:   0.8641
Macro Recall:      0.8450
Macro F1-score:    0.8408


In [12]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN BiLSTM (HANDS_POSE) ---")
bilstm_model = build_bilstm_model()

# ĐÃ XÓA: dòng bilstm_model.compile(...) vì đã được gọi bên trong hàm build_bilstm_model()

bilstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Cập nhật tên file thành bilstm_model_hands_pose.h5
    ModelCheckpoint('../models/bilstm_model_hands_pose.h5', monitor='val_loss', save_best_only=True)
]

bilstm_history = bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=bilstm_callbacks,
    verbose=1
)

# ĐÃ SỬA: Cập nhật tên file thành bilstm_history_hands_pose.json
with open('../results/bilstm_history_hands_pose.json', 'w') as f:
    json.dump(bilstm_history.history, f)

print(f"\n[*] BiLSTM training completed.")
print(f"[*] Final validation accuracy: {bilstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/bilstm_model_hands_pose.h5")


--- XÂY DỰNG & HUẤN LUYỆN BiLSTM (HANDS_POSE) ---
Epoch 1/50
94/94 [==============================] - 22s 124ms/step - loss: 2.8273 - accuracy: 0.1940 - val_loss: 1.8714 - val_accuracy: 0.5000
Epoch 2/50
 1/94 [..............................] - ETA: 7s - loss: 1.8283 - accuracy: 0.5000

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 9s 93ms/step - loss: 1.7632 - accuracy: 0.4513 - val_loss: 1.2511 - val_accuracy: 0.6571
Epoch 3/50
94/94 [==============================] - 9s 96ms/step - loss: 1.1595 - accuracy: 0.6440 - val_loss: 0.8831 - val_accuracy: 0.7476
Epoch 4/50
94/94 [==============================] - 9s 94ms/step - loss: 0.7908 - accuracy: 0.7687 - val_loss: 0.8323 - val_accuracy: 0.7810
Epoch 5/50
94/94 [==============================] - 9s 93ms/step - loss: 0.6893 - accuracy: 0.7983 - val_loss: 0.7106 - val_accuracy: 0.8190
Epoch 6/50
94/94 [==============================] - 9s 99ms/step - loss: 0.5348 - accuracy: 0.8373 - val_loss: 0.5745 - val_accuracy: 0.8333
Epoch 7/50
94/94 [==============================] - 11s 118ms/step - loss: 0.4025 - accuracy: 0.8780 - val_loss: 0.5626 - val_accuracy: 0.8333
Epoch 8/50
94/94 [==============================] - 9s 100ms/step - loss: 0.3724 - accuracy: 0.8953 - val_loss: 0.6131 - val_accuracy: 0.8476
Epoch 9/50
94/94 [===

In [13]:
# GỌI HÀM ĐÁNH GIÁ (Xuất bảng kết quả Test Set)
evaluate_and_print(bilstm_model, "BiLSTM (HANDS_POSE)", X_test, y_test)

           BILSTM (HANDS_POSE) EVALUATION RESULTS (TEST SET)
Accuracy:          89.57%
Macro Precision:   0.9138
Macro Recall:      0.8946
Macro F1-score:    0.8941


In [14]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN (HANDS_POSE) ---")
cnn1d_model = build_cnn1d_model()

# ĐÃ XÓA: dòng cnn1d_model.compile(...) vì đã được gọi bên trong hàm build_cnn1d_model()

cnn1d_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Cập nhật tên file thành cnn1d_model_hands_pose.h5
    ModelCheckpoint('../models/cnn1d_model_hands_pose.h5', monitor='val_loss', save_best_only=True)
]

cnn1d_history = cnn1d_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn1d_callbacks,
    verbose=1
)

# ĐÃ SỬA: Cập nhật tên file thành cnn1d_history_hands_pose.json
with open('../results/cnn1d_history_hands_pose.json', 'w') as f:
    json.dump(cnn1d_history.history, f)

print(f"\n[*] 1D-CNN training completed.")
print(f"[*] Final validation accuracy: {cnn1d_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/cnn1d_model_hands_pose.h5")


--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN (HANDS_POSE) ---
Epoch 1/50
94/94 [==============================] - 3s 15ms/step - loss: 2.3242 - accuracy: 0.3583 - val_loss: 2.3933 - val_accuracy: 0.3333
Epoch 2/50
10/94 [==>...........................] - ETA: 1s - loss: 1.4978 - accuracy: 0.6000

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 1s 14ms/step - loss: 1.2501 - accuracy: 0.6633 - val_loss: 1.2599 - val_accuracy: 0.6238
Epoch 3/50
94/94 [==============================] - 1s 15ms/step - loss: 0.8798 - accuracy: 0.7563 - val_loss: 1.1598 - val_accuracy: 0.6286
Epoch 4/50
94/94 [==============================] - 1s 11ms/step - loss: 0.7364 - accuracy: 0.7833 - val_loss: 1.1769 - val_accuracy: 0.6476
Epoch 5/50
94/94 [==============================] - 1s 12ms/step - loss: 0.5777 - accuracy: 0.8393 - val_loss: 0.8201 - val_accuracy: 0.7619
Epoch 6/50
94/94 [==============================] - 1s 13ms/step - loss: 0.4818 - accuracy: 0.8613 - val_loss: 0.6820 - val_accuracy: 0.7810
Epoch 7/50
94/94 [==============================] - 1s 15ms/step - loss: 0.4079 - accuracy: 0.8843 - val_loss: 0.7880 - val_accuracy: 0.7571
Epoch 8/50
94/94 [==============================] - 1s 14ms/step - loss: 0.3735 - accuracy: 0.8917 - val_loss: 0.5956 - val_accuracy: 0.8429
Epoch 9/50
94/94 [======

In [15]:
# GỌI HÀM ĐÁNH GIÁ (Xuất bảng kết quả Test Set)
evaluate_and_print(cnn1d_model, "1D-CNN (HANDS_POSE)", X_test, y_test)

           1D-CNN (HANDS_POSE) EVALUATION RESULTS (TEST SET)
Accuracy:          91.47%
Macro Precision:   0.9273
Macro Recall:      0.9115
Macro F1-score:    0.9093


In [16]:
from tensorflow.keras.models import load_model
import os # Import os để sử dụng os.path.exists

print("\n--- KIỂM TRA MODEL ĐÃ LƯU (HANDS + POSE) ---")
saved_models = {
    # ĐÃ SỬA: Cập nhật hậu tố thành _hands_pose để khớp với tên file
    "LSTM": "../models/lstm_model_hands_pose.h5",
    "BiLSTM": "../models/bilstm_model_hands_pose.h5",
    "1D-CNN": "../models/cnn1d_model_hands_pose.h5"
}

# Lấy 1 sample từ tập X_test để chạy thử
sample = X_test[:1]

for name, path in saved_models.items():
    if os.path.exists(path):
        # Tải mô hình lên
        test_model = load_model(path)
        
        # Cho mô hình dự đoán thử 1 mẫu
        pred = test_model.predict(sample, verbose=0)
        
        # Kiểm tra xem đầu ra có đúng là 30 class không
        assert pred.shape == (1, NUM_CLASSES), f"Lỗi shape ở {name}"
        
        print(f"-> [OK] {name} tải thành công. Output Shape: {pred.shape}")
    else:
        print(f"-> [LỖI] Không tìm thấy file {path}")


--- KIỂM TRA MODEL ĐÃ LƯU (HANDS + POSE) ---
-> [OK] LSTM tải thành công. Output Shape: (1, 30)
-> [OK] BiLSTM tải thành công. Output Shape: (1, 30)
-> [OK] 1D-CNN tải thành công. Output Shape: (1, 30)
